# Buổi 23 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `boosting.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Dữ liệu (mục 4.3)

In [ ]:
%matplotlib inline
import warnings

import boosting as bo
import matplotlib.pyplot as plt
import numpy as np

warnings.simplefilter("ignore")
df = bo.doc_ban_le()                 # lần đầu đọc tệp Excel mất 1–2 phút
b = bo.bang(df)
cut = bo.cac_cutoff(df)
print("số mã:", df["unique_id"].nunique(), "| ngày bán hàng:", df["t"].nunique(), "| dòng học:", len(b))
print("tỷ lệ ô bằng 0:", round(float((df["y"] == 0).mean()), 3), "| trung vị:", df["y"].median(), "| lớn nhất:", df["y"].max())

## Bước 2 — L2 và Tweedie (mục 4.2–4.3)

Khoảng 1–2 phút. Sửa `THAM_SO` rồi chạy lại ô này.

In [ ]:
tk = bo.backtest_thong_ke(df)
for m in ["SeasonalNaive", "WindowAverage", "AutoETS"]:
    print(f"{m:15s} WRMSSE {bo.wrmsse(tk, df, m):.3f}")
kq = bo.backtest(b)
print(f"LightGBM ({bo.THAM_SO['objective']}) WRMSSE {bo.wrmsse(kq, df):.3f} | số dự báo âm trước khi cắt về 0:",
      int((bo.huan_luyen(b[b['t'] <= cut[0]]).predict(b[b['t'] > cut[0]][bo.DAC_TRUNG]) < 0).sum()))

## Bước 3 — Optuna (mục 4.4)

50 lần thử, lần đầu 3–10 phút tuỳ máy; kết quả lưu vào `lab/du-lieu/cache/`. Sửa `chia_cv` rồi chạy lại ô này.

In [ ]:
b_hoc = b[b["t"] <= cut[0]].reset_index(drop=True)      # chỉ phần học trước cutoff đầu
p = bo.tune(b_hoc, 50, luu=True)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in p.items()})
p = {k: v for k, v in p.items() if k != "diem_cv"}
print(f"WRMSSE với bộ đã tune: {bo.wrmsse(bo.backtest(b, p), df):.3f}")

## Bước 4 — SHAP cho một dự báo (mục 4.5)

In [ ]:
m = bo.huan_luyen(b[b["t"] <= cut[-1]])
dong = b[(b["unique_id"] == "22659") & (b["ds"] == "2011-12-01")]
s = bo.shap_mot_dong(m, dong)
print(s.sort_values(key=abs, ascending=False).round(3).head(8).to_string())
print("tổng =", round(s.sum(), 3), "| điểm thô =", round(float(m.predict(dong[bo.DAC_TRUNG], raw_score=True)[0]), 3),
      "| dự báo =", round(float(m.predict(dong[bo.DAC_TRUNG])[0]), 1), "món | thực tế =", float(dong["y"].iloc[0]))

## Bước 5 — Partial dependence của giá (mục 4.6)

In [ ]:
X = b[(b["t"] > cut[-1]) & (b["t"] <= cut[-1] + bo.H)]
luoi = np.linspace(*np.nanquantile(b["gia"], [0.02, 0.98]), 25)
m_dd = bo.huan_luyen(b[b["t"] <= cut[-1]], bo.tham_so_don_dieu())
plt.plot(luoi, bo.phu_thuoc_rieng(m, X, "gia", luoi), marker="x", label="không ràng buộc")
plt.plot(luoi, bo.phu_thuoc_rieng(m_dd, X, "gia", luoi), marker="o", label="ràng buộc: giá tăng thì dự báo không tăng")
plt.xlabel("giá trung bình mỗi món trong 28 ngày trước (£)")
plt.ylabel("dự báo trung bình (món/ngày)")
plt.legend();

## Bước 6 — Kiểm tra

Trong terminal ở thư mục `lab/`: `python lab.py check` — phải xanh 7/7.